<a href="https://colab.research.google.com/github/kady05naik/PySpark/blob/main/dataFrameTransformations/transformations_based_on_dataframe_14.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Transformations on Data Frame 14

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [2]:
spark=SparkSession.builder \
  .appName('MyApp') \
  .getOrCreate()

In [3]:
data = [("James, A, Smith","2018","M",3000),
            ("Michael, Rose, Jones","2010","M",4000),
            ("Robert,K,Williams","2010","M",4000),
            ("Maria,Anne,Jones","2005","F",4000),
            ("Jen,Mary,Brown","2010","",-1)
            ]

In [12]:
schema=StructType([
    StructField('name', StringType(), True),
    StructField('dob_year', StringType(), True),
    StructField('gender', StringType(), True),
    StructField('salary', IntegerType(), True)
])

In [13]:
df=spark.createDataFrame(data, schema)

In [8]:
df.show()

+--------------------+--------+------+-------+
|                name|dob_year|gender|salalry|
+--------------------+--------+------+-------+
|     James, A, Smith|    2018|     M|   3000|
|Michael, Rose, Jones|    2010|     M|   4000|
|   Robert,K,Williams|    2010|     M|   4000|
|    Maria,Anne,Jones|    2005|     F|   4000|
|      Jen,Mary,Brown|    2010|      |     -1|
+--------------------+--------+------+-------+



**Do needful for below output**
```
+------------------------+--------+------+------+
|NameArray               |dob_year|gender|salary|
+------------------------+--------+------+------+
|[James,  A,  Smith]     |2018    |M     |3000  |
|[Michael,  Rose,  Jones]|2010    |M     |4000  |
|[Robert, K, Williams]   |2010    |M     |4000  |
|[Maria, Anne, Jones]    |2005    |F     |4000  |
|[Jen, Mary, Brown]      |2010    |      |-1    |
+------------------------+--------+------+------+
```



In [15]:
df.withColumn('NameArray',split(col('name'),',')) \
  .select(col('NameArray'), col('dob_year'), col('gender'), col('salary')) \
  .show(truncate=False)

+------------------------+--------+------+------+
|NameArray               |dob_year|gender|salary|
+------------------------+--------+------+------+
|[James,  A,  Smith]     |2018    |M     |3000  |
|[Michael,  Rose,  Jones]|2010    |M     |4000  |
|[Robert, K, Williams]   |2010    |M     |4000  |
|[Maria, Anne, Jones]    |2005    |F     |4000  |
|[Jen, Mary, Brown]      |2010    |      |-1    |
+------------------------+--------+------+------+





---



**Do needful for below output**


```
+-------+------+
|name   |col   |
+-------+------+
|James  |Java  |
|James  |Scala |
|James  |C++   |
|Michael|Spark |
|Michael|Java  |
|Michael|C++   |
|Robert |CSharp|
|Robert |VB    |
+-------+------+
```



In [22]:
arrayArrayData = [
  ("James",["Java","Scala","C++"]),
  ("Michael",["Spark","Java","C++"]),
  ("Robert",["CSharp","VB"])
]

In [23]:
schema=StructType([
    StructField('name', StringType(), True),
    StructField('skills', ArrayType( StringType()), True)
])

In [24]:
df=spark.createDataFrame(arrayArrayData, schema)

In [25]:
df.show()

+-------+------------------+
|   name|            skills|
+-------+------------------+
|  James|[Java, Scala, C++]|
|Michael|[Spark, Java, C++]|
| Robert|      [CSharp, VB]|
+-------+------------------+



### **explode() - creates one row per element**

In [31]:
df.withColumn('skills', explode('skills')).show()

+-------+------+
|   name|skills|
+-------+------+
|  James|  Java|
|  James| Scala|
|  James|   C++|
|Michael| Spark|
|Michael|  Java|
|Michael|   C++|
| Robert|CSharp|
| Robert|    VB|
+-------+------+





---



**Do needful for below output**

```
+-------+-------+
|name   |subject|
+-------+-------+
|James  |Java   |
|James  |Scala  |
|James  |C++    |
|James  |Spark  |
|James  |Java   |
|Michael|Spark  |
|Michael|Java   |
|Michael|C++    |
|Michael|Spark  |
|Michael|Java   |
|Robert |CSharp |
|Robert |VB     |
|Robert |Spark  |
|Robert |Python |
+-------+-------+
```



In [34]:
arrayArrayData = [
  ("James",[["Java","Scala","C++"],["Spark","Java"]]),
  ("Michael",[["Spark","Java","C++"],["Spark","Java"]]),
  ("Robert",[["CSharp","VB"],["Spark","Python"]])
]

In [70]:
schema=StructType([
    StructField('name', StringType(), True),
    StructField('subject', ArrayType(ArrayType(StringType()),True))
])

In [71]:
df=spark.createDataFrame(arrayArrayData, schema)

In [40]:
df.show(truncate=False)

+-------+-----------------------------------+
|name   |subject                            |
+-------+-----------------------------------+
|James  |[[Java, Scala, C++], [Spark, Java]]|
|Michael|[[Spark, Java, C++], [Spark, Java]]|
|Robert |[[CSharp, VB], [Spark, Python]]    |
+-------+-----------------------------------+



## **Since the column is an array of arrays, flatten() first converts it into a single array and then explode() turns each element into a row. This is usually the most optimized and readable solution compared with doing explode() twice.**

In [84]:
df.withColumn('subjects',explode(flatten(col('subject')))) \
  .select('name','subjects') \
  .show(truncate=False)

+-------+--------+
|name   |subjects|
+-------+--------+
|James  |Java    |
|James  |Scala   |
|James  |C++     |
|James  |Spark   |
|James  |Java    |
|Michael|Spark   |
|Michael|Java    |
|Michael|C++     |
|Michael|Spark   |
|Michael|Java    |
|Robert |CSharp  |
|Robert |VB      |
|Robert |Spark   |
|Robert |Python  |
+-------+--------+



**flatten() removes one nesting level from an array-of-arrays in one expression.**



---



In [82]:
df.select('name',explode(col('subject')).alias('arr'))\
  .select('name',explode(col('arr'))) \
  .show()

+-------+------------------+
|   name|               arr|
+-------+------------------+
|  James|[Java, Scala, C++]|
|  James|     [Spark, Java]|
|Michael|[Spark, Java, C++]|
|Michael|     [Spark, Java]|
| Robert|      [CSharp, VB]|
| Robert|   [Spark, Python]|
+-------+------------------+



**explode() returns one row per array element. Spark allows only one explode per SELECT clause, so the two-explode version needs two projection steps, which makes the logical plan a bit more verbose.**



---



## **how would you create the datadframe from nested json . please create a function to read the nested json data**

**To create a DataFrame from nested JSON in PySpark, I use spark.read.json(path). If the file is a regular multi-line JSON file or a JSON array, I set option("multiLine", True). For production jobs, I prefer passing an explicit StructType schema to avoid Spark’s extra schema inference scan. Nested JSON objects become struct columns and nested lists become array columns, which I can later access with dot notation or flatten using explode()**

In [85]:
spark.stop()